# Notebook 09 — Hybrydowy Indeks Ryzyka (HRI)

**Projekt:** Współzależność rynków aktywów cyfrowych i tradycyjnych systemów finansowych  

---

## Cel notebooka

Analizujemy **Hybrydowy Indeks Ryzyka (HRI)** — wyjście systemu Ensemble Stacking:

- **HRI ∈ [0,1]**: prawdopodobieństwo zaburzeń płynności WIG20 w ciągu 5 dni
- **Progi interpretacyjne:**
  - HRI < 0.3: niskie ryzyko
  - HRI ∈ [0.3, 0.5]: umiarkowane ryzyko
  - HRI ∈ [0.5, 0.7]: podwyższone ryzyko  
  - HRI > 0.7: **wysokie ryzyko** — sygnał ostrzegawczy

## Kluczowe analizy
1. Wizualizacja HRI w czasie 2019-2024
2. Scatter HRI(t) vs zwrot WIG20(t+5) — czy HRI przewiduje spadki?
3. Zbliżenia na kryzysy: COVID, Terra/LUNA, FTX
4. Test Grangera: HRI → WIG20
5. Event study: skumulowany zwrot WIG20 po przekroczeniu HRI > 0.7

In [1]:
# ============================================================
# Konfiguracja wspolna
# ============================================================
import os, warnings
os.environ.setdefault('KERAS_BACKEND', 'torch')
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')
warnings.filterwarnings('ignore')
import importlib
import pandas as pd, numpy as np
import pipeline_lib as pl, models_lib as ml, run_analysis as ra
for m in (pl, ml, ra): importlib.reload(m)
pd.set_option('display.width', 160)
print(f'Ziarno losowe: {pl.SEED} | horyzont: {pl.HORIZON} sesji | embargo: {pl.EMBARGO} sesji')


Ziarno losowe: 42 | horyzont: 5 sesji | embargo: 10 sesji


In [2]:
# ============================================================
# Hybrydowy Indeks Ryzyka: prog, epizody krytyczne, transmisja
# ============================================================
oof, y, master = ra.load()
res = ra.hri_analysis(oof, y, master)
print(f'Prog ostrzegawczy (kwantyl {ra.HRI_QUANTILE} okresu referencyjnego): {res["threshold"]:.4f}')
print(f'Liczba epizodow krytycznych: {len(res["periods"])}\n')
print(res['periods'].to_string(index=False))
print('\nTest przyczynowosci Grangera (wartosci p):')
print(res['granger'].to_string())
print('\nAnaliza zdarzen — skumulowane ponadprzecietne stopy zwrotu:')
car = res['car']
print(car[car.day_relative.isin([-10,-5,-3,-1,0,1,3,5,7,10])].to_string(index=False))


Prog ostrzegawczy (kwantyl 0.9 okresu referencyjnego): 0.2498
Liczba epizodow krytycznych: 4

Data początku Data końca  Liczba sesji  Czas trwania (dni kalendarzowe)  Maks. HRI  Średni HRI
   2022-10-19 2022-10-31             9                               13     0.2856      0.2679
   2022-11-17 2022-12-13            19                               27     0.2916      0.2711
   2024-07-18 2024-07-29             8                               12     0.3692      0.3369
   2025-04-07 2025-04-24            12                               18     0.9668      0.6263

Test przyczynowosci Grangera (wartosci p):
     HRI → WIG20  WIG20 → HRI
Lag                          
1        0.01978      0.34329
2        0.08083      0.96341
3        0.01267      0.27492
4        0.02576      0.80370
5        0.04684      0.36118
6        0.05353      0.75869
7        0.06162      0.94083
8        0.08128      0.25001
9        0.09410      0.24144
10       0.12398      0.12128

Analiza zdarzen — skumulow